# xLSTM GPU Training — Riccio (angles) + EgoExo (CLIP)

This notebook trains both models on Colab GPU:
1. **Riccio classifier** on joint angles (8-dim, camera-invariant) — generalizes better
2. **EgoExo quality model** on CLIP ViT-B/32 features (512-dim) — quality + error tags
3. **Ablation** comparing CLIP vs annotation features

## Setup
- Runtime → Change runtime type → **GPU (T4)**
- Upload your data or mount Google Drive

In [1]:
#@title 1. Setup environment
!pip install torch torchvision torchaudio -q
!pip install scikit-learn mediapipe opencv-python-headless -q

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 100.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 16.0 MB/s eta 0:00:00
PyTorch 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
#@title 2. Clone repo
import os

REPO = "https://github.com/Elina425/Finess-coach-capstone.git"
BRANCH = "trial1"
WORKSPACE = "/content/Finess-coach-capstone-1"

if not os.path.exists(WORKSPACE):
    !git clone -b {BRANCH} {REPO} {WORKSPACE}
%cd {WORKSPACE}
!pip install -e . -q 2>/dev/null || echo 'No setup.py, using sys.path'

import sys
if WORKSPACE not in sys.path:
    sys.path.insert(0, WORKSPACE)

In [ ]:
#@title 3. Mount Google Drive (for saving results)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_RESULTS = '/content/drive/MyDrive/fitness_coach_results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print(f'Results will be saved to: {DRIVE_RESULTS}')

## Part A: Upload Riccio NPZ files

Upload your 3 files from `results/riccio_realtime_exercise_recognition/`:
- `riccio_realtime_exercise_recognition_biomechanics.npz`
- `riccio_realtime_exercise_recognition_keypoints.npz`
- `riccio_realtime_exercise_recognition_labels.npz`

Either upload via the file browser or copy from Drive.

In [ ]:
#@title 4a. Copy Riccio NPZs from Drive (edit path if needed)
import shutil

RICCIO_DRIVE = '/content/drive/MyDrive/riccio_realtime_exercise_recognition'  #@param {type:"string"}
RICCIO_LOCAL = f'{WORKSPACE}/results/riccio_realtime_exercise_recognition'
os.makedirs(RICCIO_LOCAL, exist_ok=True)

for f in ['riccio_realtime_exercise_recognition_biomechanics.npz',
          'riccio_realtime_exercise_recognition_keypoints.npz',
          'riccio_realtime_exercise_recognition_labels.npz']:
    src = os.path.join(RICCIO_DRIVE, f)
    dst = os.path.join(RICCIO_LOCAL, f)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f'Copied {f}')
    elif os.path.exists(dst):
        print(f'Already exists: {f}')
    else:
        print(f'NOT FOUND: {src}')

# OR upload manually:
# from google.colab import files
# uploaded = files.upload()  # then move to RICCIO_LOCAL

In [ ]:
#@title 4b. Verify Riccio data
import numpy as np

ang = np.load(f'{RICCIO_LOCAL}/riccio_realtime_exercise_recognition_biomechanics.npz')
kp = np.load(f'{RICCIO_LOCAL}/riccio_realtime_exercise_recognition_keypoints.npz')
lab = np.load(f'{RICCIO_LOCAL}/riccio_realtime_exercise_recognition_labels.npz', allow_pickle=True)
print(f'Angles: {ang["angles"].shape}')  # (T, 8)
print(f'Keypoints: {kp["keypoints"].shape}')  # (T, 17, 2)
print(f'Labels: {len(lab["pose"])} frames')

## Part B: Train Riccio xLSTM classifier (angles, 8-dim)

Joint angles are camera-invariant — a 90° knee bend is 90° regardless of
camera position. This generalizes much better than raw (x,y) coordinates.

In [ ]:
#@title 5. Train Riccio angles classifier (GPU)
!python3 train_xlstm_keypoints.py \
  --kaggle-keypoints-dir results/riccio_realtime_exercise_recognition \
  --kaggle-stem riccio_realtime_exercise_recognition \
  --feature-mode angles \
  --output-dir results/xlstm_riccio_angles \
  --eval-test \
  --epochs 100 \
  --hidden 128 \
  --layers 3

In [ ]:
#@title 5b. Also train coords model for ablation comparison
!python3 train_xlstm_keypoints.py \
  --kaggle-keypoints-dir results/riccio_realtime_exercise_recognition \
  --kaggle-stem riccio_realtime_exercise_recognition \
  --feature-mode coords \
  --output-dir results/xlstm_riccio_coords \
  --eval-test \
  --epochs 100 \
  --hidden 128 \
  --layers 3

In [ ]:
#@title 5c. And mixed (angles + coords) for ablation
!python3 train_xlstm_keypoints.py \
  --kaggle-keypoints-dir results/riccio_realtime_exercise_recognition \
  --kaggle-stem riccio_realtime_exercise_recognition \
  --feature-mode mixed \
  --output-dir results/xlstm_riccio_mixed \
  --eval-test \
  --epochs 100 \
  --hidden 128 \
  --layers 3

In [ ]:
#@title 6. Compare ablation results
import json

ablation = {}
for mode in ['angles', 'coords', 'mixed']:
    path = f'results/xlstm_riccio_{mode}/test_metrics.json'
    if os.path.exists(path):
        with open(path) as f:
            m = json.load(f)
        ablation[mode] = {
            'accuracy': m.get('accuracy', 0),
            'f1_macro': m.get('f1_macro', 0),
            'f1_weighted': m.get('f1_weighted', 0),
        }
        print(f'{mode:8s}: acc={m["accuracy"]:.4f}  F1_macro={m["f1_macro"]:.4f}  F1_weighted={m["f1_weighted"]:.4f}')

with open('results/riccio_ablation_comparison.json', 'w') as f:
    json.dump(ablation, f, indent=2)
print('\nSaved to results/riccio_ablation_comparison.json')

## Part C: Upload EgoExo data and train CLIP quality model

Upload the EgoExo index CSV and CLIP feature .pth files.

In [ ]:
#@title 7a. Copy EgoExo data from Drive
EGOEXO_DRIVE = '/content/drive/MyDrive/egoexo_fitness_full'  #@param {type:"string"}

# Copy index CSV
os.makedirs(f'{WORKSPACE}/results', exist_ok=True)
for csv_name in ['egoexo_fitness_index_split.csv', 'egoexo_fitness_index.csv']:
    src = os.path.join(EGOEXO_DRIVE, csv_name)
    dst = f'{WORKSPACE}/results/{csv_name}'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f'Copied {csv_name}')

# Symlink CLIP features (avoid copying ~2GB)
CLIP_SRC = os.path.join(EGOEXO_DRIVE, 'features_open/visual')
CLIP_DST = f'{WORKSPACE}/notebooks/data/egoexo_fitness_full/features_open/visual'
os.makedirs(os.path.dirname(CLIP_DST), exist_ok=True)
if os.path.exists(CLIP_SRC) and not os.path.exists(CLIP_DST):
    os.symlink(CLIP_SRC, CLIP_DST)
    print(f'Symlinked CLIP features')

# Also copy raw annotations for index rebuild if needed
ANN_SRC = os.path.join(EGOEXO_DRIVE, 'raw_annotations')
ANN_DST = f'{WORKSPACE}/data/EgoExo-Fitness/raw_annotations'
if os.path.exists(ANN_SRC) and not os.path.exists(ANN_DST):
    os.makedirs(os.path.dirname(ANN_DST), exist_ok=True)
    os.symlink(ANN_SRC, ANN_DST)
    print(f'Symlinked annotations')

In [ ]:
#@title 7b. Rebuild EgoExo index if needed
if not os.path.exists(f'{WORKSPACE}/results/egoexo_fitness_index_split.csv'):
    !python3 build_egoexo_fitness_index.py \
      --annotations-json data/EgoExo-Fitness/raw_annotations/interpretable_action_judgement.json \
      --dataset-root data/EgoExo-Fitness \
      --format interpretable \
      --quality-scale 1-5 \
      --output results/egoexo_fitness_index.csv
    !python3 split_exercise_index.py \
      --input results/egoexo_fitness_index.csv \
      --output results/egoexo_fitness_index_split.csv
else:
    print('EgoExo index already exists')

In [ ]:
#@title 8. Train EgoExo CLIP quality model (GPU)
!python3 train_xlstm_egoexo_multitask.py \
  --index-csv results/egoexo_fitness_index_split.csv \
  --feature-mode clip \
  --clip-features-root notebooks/data/egoexo_fitness_full/features_open/visual \
  --output-dir results/xlstm_egoexo_clip \
  --standardize --eval-test \
  --epochs 60 \
  --hidden 128 \
  --layers 4

In [ ]:
#@title 9. View all results
import json

print('='*60)
print('RICCIO CLASSIFIER RESULTS (angles vs coords vs mixed)')
print('='*60)
for mode in ['angles', 'coords', 'mixed']:
    path = f'results/xlstm_riccio_{mode}/test_metrics.json'
    if os.path.exists(path):
        m = json.load(open(path))
        print(f'\n{mode.upper()}:')
        print(f'  Accuracy:    {m["accuracy"]:.4f}')
        print(f'  F1 macro:    {m["f1_macro"]:.4f}')
        print(f'  F1 weighted: {m["f1_weighted"]:.4f}')
        if 'f1_per_class' in m:
            names = m.get('class_names', [])
            vals = m['f1_per_class']
            if isinstance(vals, dict):
                for k, v in vals.items():
                    print(f'    {k}: {v:.4f}')
            elif isinstance(vals, list) and names:
                for n, v in zip(names, vals):
                    print(f'    {n}: {v:.4f}')

print()
print('='*60)
print('EGOEXO CLIP QUALITY MODEL RESULTS')
print('='*60)
clip_path = 'results/xlstm_egoexo_clip/metrics.json'
if os.path.exists(clip_path):
    m = json.load(open(clip_path))
    for split in ['best_val', 'test']:
        if m.get(split):
            s = m[split]
            print(f'\n{split.upper()}:')
            print(f'  Classification accuracy: {s["accuracy"]:.4f}')
            print(f'  F1 macro:    {s["f1_macro"]:.4f}')
            print(f'  Quality MAE: {s["mae"]:.4f}')
            print(f'  Quality R2:  {s["r2"]:.4f}')
            print(f'  Error F1:    {s["error_f1_macro"]:.4f}')

In [ ]:
#@title 10. Copy results to Drive
import shutil

for folder in ['xlstm_riccio_angles', 'xlstm_riccio_coords', 'xlstm_riccio_mixed', 'xlstm_egoexo_clip']:
    src = f'results/{folder}'
    dst = f'{DRIVE_RESULTS}/{folder}'
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'Copied {folder} to Drive')

# Also copy the ablation comparison
abl = 'results/riccio_ablation_comparison.json'
if os.path.exists(abl):
    shutil.copy2(abl, DRIVE_RESULTS)

print(f'\nAll results saved to {DRIVE_RESULTS}')
print('Download the angles checkpoint to use locally:')
print(f'  {DRIVE_RESULTS}/xlstm_riccio_angles/xlstm_keypoints_best.pt')